# 01 — Learn the Canadian Nutrient File (CNF) 2026

A guided, part-by-part tour of the database behind this app. Run one
section at a time, top to bottom. Each section teaches one idea with
working pandas code, then ends with a **Your turn** cell where you
modify the code to answer a question yourself.

**Who this is for:** a registered dietitian who is learning Python and
pandas, and who keeps getting asked "what's in the CNF?" by AI models
and colleagues. By the end you will have answered that question with
your own hands.

**What the CNF is:** a public Government of Canada dataset of
~5,993 foods × ~173 nutrients, with every nutrient value expressed
**per 100 g of edible food**. It is delivered as a set of CSV files
that together form a *relational database* — we'll unpack that term in
Part 1.

**How to use this notebook:** select a code cell and press
`Shift+Enter` to run it. Read the markdown between cells for context.
When you hit a **Your turn** cell, try to write the code before
scrolling to the Answers section at the bottom.

## Part 0 — Setup & the BOM lesson

Before we explore the data, one technical detail that will bite you if
you don't know it: **several CNF CSV files start with a UTF-8 BOM** (Byte
Order Mark — an invisible character `\ufeff` at the very start of the
file). If you read such a file with plain `encoding="utf-8"`, pandas
keeps the BOM glued to the first column name, so `Nutrient_Code`
becomes `\ufeffNutrient_Code` — and any merge that looks for
`"Nutrient_Code"` silently finds nothing.

The fix is `encoding="utf-8-sig"`, which strips the BOM.

Let's see the bug and the fix side by side.

In [1]:
import pandas as pd
from pathlib import Path

# All raw CNF CSVs live here, relative to this notebook:
DATA_DIR = Path("../cnf_fcen_all-files-data_2026")

# --- The WRONG way: plain utf-8 on a BOM file ---
nutrient_name_buggy = pd.read_csv(DATA_DIR / "Nutrient_Name.csv", encoding="utf-8")
print("First column name (buggy):", repr(nutrient_name_buggy.columns[0]))
print("  → Notice the \\ufeff prefix. A merge on 'Nutrient_Code' would fail.")

First column name (buggy): 'Nutrient_Code'
  → Notice the \ufeff prefix. A merge on 'Nutrient_Code' would fail.


In [2]:
# --- The RIGHT way: utf-8-sig strips the BOM ---
nutrient_name = pd.read_csv(DATA_DIR / "Nutrient_Name.csv", encoding="utf-8-sig")
print("First column name (fixed):", repr(nutrient_name.columns[0]))
print("  → Clean. Merges will work.")

First column name (fixed): 'Nutrient_Code'
  → Clean. Merges will work.


**Which files have the BOM?** From experience in this project:
`Nutrient_Name.csv`, `Nutrient_Amount.csv`, `Measure_Name.csv`,
`Measure_Weight_Conversion.csv`, and `Food_Source.csv` all carry it.
`Food_Name.csv` does **not** — plain `utf-8` works fine for that one.

A safe habit: use `encoding="utf-8-sig"` for every CNF file. It works
on files with a BOM *and* on files without one, so you never have to
remember which is which.

### Your turn 0

Load `Food_Name.csv` into a DataFrame called `food_name`. This file has
**no** BOM, so both `utf-8` and `utf-8-sig` work — but use `utf-8-sig`
anyway to build the habit. Print its shape (rows × columns) and the
first column name to confirm it loaded cleanly.

In [3]:
# Your code here:
# food_name = pd.read_csv(...)
# print(...)

## Part 1 — The map: 9 files, one database

The CNF is a **relational database** delivered as CSVs. That means the
data is split across multiple tables, linked by shared ID columns,
rather than crammed into one giant spreadsheet. Two terms you need:

- **Primary key (PK):** the column that uniquely identifies a row in a
  table — like a student number for a student.
- **Foreign key (FK):** a column in one table that points to another
  table's primary key — like a `course_id` column in an enrolments
  table.

The reason for splitting data this way is **normalization**: store each
fact once, in one place, and link to it. If a food's name changed, you'd
update one row in `Food_Name.csv`, not 565,000 rows in
`Nutrient_Amount.csv`.

Here are all 9 files and how they connect:

```
Food_Name.csv          Food_Code (PK) ─────────┐
  │                                            │
  │  CNF_Food_Group_Code ──► CNF_Food_Group.csv (PK)
  │  Food_Source_Code    ──► Food_Source.csv   (PK)
  │                                            │
  ▼                                            │
Nutrient_Amount.csv    Food_Code (FK) ─────────┘
                       Nutrient_Code (FK) ──► Nutrient_Name.csv (PK)
                       Nutrient_Source_Code ──► Nutrient_Source.csv (PK)

Measure_Weight_Conversion.csv
        Food_Code (FK) ──► Food_Name.csv
        Measure_Code (FK) ──► Measure_Name.csv (PK)
        Measure_Type_Code ──► Measure_Type.csv (PK)
```

**The one rule that governs everything:** all nutrient amounts in
`Nutrient_Amount.csv` are expressed **per 100 g of edible food**. Every
calculation in the app — every kcal, every gram of protein — starts
from that convention.

In [4]:
# Load all 9 CSVs. We use utf-8-sig everywhere (safe for both BOM and
# non-BOM files). The app's own loader (src/data_loader.py) does the same
# thing, but here you're doing it by hand so you see the mechanics.

tables = {}
tables["food_name"]              = pd.read_csv(DATA_DIR / "Food_Name.csv", encoding="utf-8-sig")
tables["nutrient_name"]          = pd.read_csv(DATA_DIR / "Nutrient_Name.csv", encoding="utf-8-sig")
tables["nutrient_amount"]        = pd.read_csv(DATA_DIR / "Nutrient_Amount.csv", encoding="utf-8-sig")
tables["measure_name"]           = pd.read_csv(DATA_DIR / "Measure_Name.csv", encoding="utf-8-sig")
tables["measure_type"]           = pd.read_csv(DATA_DIR / "Measure_Type.csv", encoding="utf-8-sig")
tables["measure_weight_conversion"] = pd.read_csv(DATA_DIR / "Measure_Weight_Conversion.csv", encoding="utf-8-sig")
tables["food_group"]             = pd.read_csv(DATA_DIR / "CNF_Food_Group.csv", encoding="utf-8-sig")
tables["nutrient_source"]        = pd.read_csv(DATA_DIR / "Nutrient_Source.csv", encoding="utf-8-sig")
tables["food_source"]            = pd.read_csv(DATA_DIR / "Food_Source.csv", encoding="utf-8-sig")

for name, df in tables.items():
    print(f"{name:30s}  {df.shape[0]:>8,} rows × {df.shape[1]} cols")

food_name                          5,993 rows × 12 cols
nutrient_name                        173 rows × 7 cols
nutrient_amount                  565,409 rows × 7 cols
measure_name                       1,494 rows × 3 cols
measure_type                           3 rows × 3 cols
measure_weight_conversion         29,868 rows × 5 cols
food_group                            23 rows × 3 cols
nutrient_source                       21 rows × 3 cols
food_source                           16 rows × 3 cols


### Your turn 1

Two of these tables share a column called `Nutrient_Source_Code`. Which
two? Inspect the columns of each table (hint: `df.columns` gives you
the list) and write them below.

In [5]:
# Your code here:
# for name, df in tables.items():
#     if "Nutrient_Source_Code" in df.columns:
#         print(name)

## Part 2 — Food_Name: the catalogue

`Food_Name.csv` is the master list of ~5,993 foods. Each row is one
food, identified by its `Food_Code` (the primary key). The most useful
columns for us:

- `Food_Description_EN` — the English name (e.g. "Chicken, broiler,
  breast, meat and skin, roasted")
- `CNF_Food_Group_Code` — links to `CNF_Food_Group.csv` (dairy,
  vegetables, meats, etc.)
- `USDA_NDB_Code` — a cross-reference to the USDA database (more on
  this in Part 6)

**A quirk that matters:** CNF names foods the way a librarian files
them, not the way you say them. Wild rice is stored as
"Grains, rice, wild, dry". Greek yogurt is "Yogourt, Greek, ...". The
most important word often comes *last*. This is why a naive search for
"wild rice" finds nothing — and it's the reason this app has a
three-layer search module (`src/food_search.py`) instead of a simple
substring match.

In [6]:
food_name = tables["food_name"]

# Case-insensitive substring search. regex=False is important: it treats
# the search string as literal text, so characters like "(" or "." in a
# food name don't break the search.
chicken = food_name[food_name["Food_Description_EN"].str.contains(
    "chicken", case=False, na=False, regex=False
)]
print(f"Foods containing 'chicken': {len(chicken)}")
chicken[["Food_Code", "Food_Description_EN"]].head(10)

Foods containing 'chicken': 344


,Food_Code,Food_Description_EN
2,5,"Chinese dish, chow mein, chicken"
5,8,"Frozen entree, fried chicken with mashed potat..."
68,83,"Egg, chicken, dried, whole"
69,84,"Egg, chicken, dried, whole, stabilized"
70,85,"Egg, chicken, white, pan dried, flakes"
71,86,"Egg, chicken, white, dried, powder, glucose re..."
72,87,"Egg, chicken, yolk, dried"
105,125,"Egg, chicken, whole, fresh or frozen, raw"
106,126,"Egg, chicken, white, fresh or frozen, raw"
107,127,"Egg, chicken, yolk, fresh or frozen, raw"


In [7]:
# The "wild rice" trap — let's see it ourselves:
wild_rice_naive = food_name[food_name["Food_Description_EN"].str.contains(
    "wild rice", case=False, na=False, regex=False
)]
print(f"Searching 'wild rice' (naive): {len(wild_rice_naive)} results")

# But search for just "wild":
wild = food_name[food_name["Food_Description_EN"].str.contains(
    "wild", case=False, na=False, regex=False
)]
print(f"Searching 'wild': {len(wild)} results")
wild[["Food_Code", "Food_Description_EN"]].head(5)

Searching 'wild rice' (naive): 0 results
Searching 'wild': 55 results


,Food_Code,Food_Description_EN
354,667,"Duck, wild, Indigenous, meat and skin, raw"
355,668,"Duck, wild, Indigenous, breast, meat only, raw"
2078,2993,"Fish, catfish, channel (bullhead), wild, raw"
2079,2994,"Fish, catfish, channel (bullhead), wild, bread..."
2134,3049,"Fish, salmon, atlantic, wild, raw"


In [8]:
# Join to food_group to see which categories the foods fall into.
# A "join" (also called "merge") combines two tables on a shared key —
# think VLOOKUP in Excel. Here we look up each food's group description.
food_with_group = food_name.merge(
    tables["food_group"],
    on="CNF_Food_Group_Code",
    how="left",  # keep all foods, even if their group code is missing
)

# Count foods per group, sorted descending:
group_counts = food_with_group["CNF_Food_Group_Description_EN"].value_counts()
print("Foods per group (top 10):")
print(group_counts.head(10))

Foods per group (top 10):
CNF_Food_Group_Description_EN
Vegetables and Vegetable Products    790
Baked Products                       573
Poultry Products                     419
Lamb, Veal and Game                  363
Fruits and fruit juices              340
Sweets                               326
Finfish and Shellfish Products       325
Dairy and Egg Products               296
Beverages                            284
Soups, Sauces and Gravies            279
Name: count, dtype: int64


### Your turn 2

How many foods contain "cheese" in their English description? And which
food group do most of them belong to?

In [9]:
# Your code here:
# cheese = food_name[food_name["Food_Description_EN"].str.contains(...)]
# print(f"Count: {len(cheese)}")
# # Then merge with food_group and count...

## Part 3 — Nutrient_Name: the 173 nutrients

`Nutrient_Name.csv` defines the ~173 nutrients CNF tracks. Each row is
one nutrient, identified by `Nutrient_Code` (the primary key). Useful
columns:

- `Nutrient_Name_EN` — e.g. "Protein", "Fat (total lipids)"
- `Nutrient_Symbol` — a short code like "PROT", "FAT"
- `Nutrient_Unit` — "Gram", "mg", "kcal", "µg"
- `Tagname` — a standardized abbreviation used internationally

The app doesn't display all 173. It tracks 19 (defined in
`data/packs/canada/nutrients.csv`) — the ones on a Canadian Nutrition
Facts panel, plus a few clinical extras. CNF tracks far more than any
app shows.

In [10]:
nutrient_name = tables["nutrient_name"]
print(f"Total nutrients in CNF: {len(nutrient_name)}")
print(f"Columns: {list(nutrient_name.columns)}")
nutrient_name[["Nutrient_Code", "Nutrient_Symbol", "Nutrient_Unit", "Nutrient_Name_EN"]].head(10)

Total nutrients in CNF: 173
Columns: ['Nutrient_Code', 'Nutrient_Symbol', 'Nutrient_Unit', 'Nutrient_Name_EN', 'Nutrient_Name_FR', 'Tagname', 'Nutrient_Decimals']


,Nutrient_Code,Nutrient_Symbol,Nutrient_Unit,Nutrient_Name_EN
0,203,PROT,Gram,Protein
1,204,FAT,Gram,Fat (total lipids)
2,205,CARB,Gram,"Carbohydrate, total (by difference)"
3,207,ASH,Gram,"Ash, total"
4,208,KCAL,kilocalorie,Energy (kilocalories)
5,210,SUCR,Gram,Sucrose
6,211,GLUC,Gram,Glucose
7,212,FRUC,Gram,Fructose
8,213,LACT,Gram,Lactose
9,214,MALT,Gram,Maltose


In [11]:
# Find the key nutrients the app cares about, by name:
key_names = [
    "Protein", "Fat (total lipids)", "Carbohydrate", "Energy",
    "Water (moisture)", "Fibre", "Sodium", "Potassium", "Calcium", "Iron",
]

key_nutrients = nutrient_name[nutrient_name["Nutrient_Name_EN"].isin(key_names)]
key_nutrients[["Nutrient_Code", "Nutrient_Symbol", "Nutrient_Unit", "Nutrient_Name_EN"]]

,Nutrient_Code,Nutrient_Symbol,Nutrient_Unit,Nutrient_Name_EN
0,203,PROT,Gram,Protein
1,204,FAT,Gram,Fat (total lipids)
20,301,CA,Milligram,Calcium
21,303,FE,Milligram,Iron
24,306,K,Milligram,Potassium
25,307,NaN,Milligram,Sodium


In [12]:
# Compare against the app's own registry — 19 nutrients, not 173.
# The app's registry lives in data/packs/canada/nutrients.csv.
registry = pd.read_csv("../data/packs/canada/nutrients.csv")
print(f"App tracks {len(registry)} nutrients:")
print(registry[["name", "code", "label", "unit", "tier"]].to_string(index=False))

App tracks 19 nutrients:
           name  code            label unit     tier
    energy_kcal   208           Energy kcal    label
          fat_g   204              Fat    g    label
saturated_fat_g   606    Saturated Fat    g    label
    trans_fat_g   605        Trans Fat    g    label
 carbohydrate_g   205     Carbohydrate    g    label
        fibre_g   291            Fibre    g    label
       sugars_g   269           Sugars    g    label
      protein_g   203          Protein    g    label
 cholesterol_mg   601      Cholesterol   mg    label
      sodium_mg   307           Sodium   mg    label
   potassium_mg   306        Potassium   mg    label
     calcium_mg   301          Calcium   mg    label
        iron_mg   303             Iron   mg    label
        water_g   255 Water (moisture)    g   engine
   magnesium_mg   304        Magnesium   mg clinical
  phosphorus_mg   305       Phosphorus   mg clinical
        zinc_mg   309             Zinc   mg clinical
   vitamin_d_ug   328

In [13]:
# Integrity check: do all the app's nutrient codes actually exist in CNF?
# (If one didn't, the app would silently compute zero for that nutrient.)
registry_codes = set(registry["code"])
cnf_codes = set(nutrient_name["Nutrient_Code"])
missing = registry_codes - cnf_codes
if missing:
    print(f"⚠️  Codes in the app registry but NOT in CNF: {missing}")
else:
    print("✅ Every code in the app's registry exists in CNF.")

✅ Every code in the app's registry exists in CNF.


### Your turn 3

Find the `Nutrient_Code` and `Nutrient_Unit` for the nutrient whose
English name contains "Sugars" (hint: use `str.contains`).

In [14]:
# Your code here:
# sugars = nutrient_name[nutrient_name["Nutrient_Name_EN"].str.contains(...)]
# print(sugars[["Nutrient_Code", "Nutrient_Name_EN", "Nutrient_Unit"]])

## Part 4 — Nutrient_Amount: the 565k-row heart

This is the biggest table and the one the app uses most. Each row is
one **food–nutrient pair**: "Food X has Y amount of nutrient Z, per
100 g". With ~5,993 foods × ~173 nutrients, that's ~565,000 rows.

This format is called **long format** (or "narrow" format): one row
per observation, with the nutrient identity stored in a column rather
than spread across 173 columns. It's the efficient way to store sparse
data — not every food has every nutrient measured. The alternative,
**wide format** (one column per nutrient), is what humans like to read,
and we'll convert to it later with `pivot`.

**Missing = absent row, not zero.** If a food wasn't analysed for a
nutrient, there is simply no row for that food–nutrient pair. A zero
would mean "measured and found to be zero" — a different statement.
This distinction is why the app shows a "Coverage" column.

In [15]:
nutrient_amount = tables["nutrient_amount"]
print(f"Shape: {nutrient_amount.shape}")
print(f"Columns: {list(nutrient_amount.columns)}")
nutrient_amount.head(10)

Shape: (565409, 7)
Columns: ['Food_Code', 'Nutrient_Code', 'Nutrient_Amount', 'STD_Error', 'Observations', 'Nutrient_Source_Code', 'Nutrient_Last_Updated_Date']


,Food_Code,Nutrient_Code,Nutrient_Amount,STD_Error,Observations,Nutrient_Source_Code,Nutrient_Last_Updated_Date
0,2,203,9.54415,0.0,0.0,51,2010-04-16
1,2,204,15.70470,0.0,0.0,51,2010-04-16
2,2,205,5.91100,0.0,0.0,51,2010-04-16
3,2,207,1.67400,0.0,0.0,51,2010-04-16
4,2,208,204.00000,0.0,0.0,51,2010-04-16
5,2,221,0.00000,NaN,0.0,12,2004-09-13
6,2,255,67.16500,0.0,0.0,51,2010-04-16
7,2,262,0.00000,NaN,0.0,12,2004-09-13
8,2,263,0.00000,NaN,0.0,12,2004-09-13
9,2,268,853.00000,0.0,0.0,4,2010-04-16


### A merge: building a readable nutrient panel for one food

To see "what nutrients does this chicken have?", we need to combine
`Nutrient_Amount` (which has codes and values) with `Nutrient_Name`
(which has the human-readable names). This is a **merge** — the pandas
equivalent of a SQL JOIN or an Excel VLOOKUP.

In [16]:
# Pick a food: "Chicken, broiler, breast, meat and skin, roasted"
food_name = tables["food_name"]
chicken_match = food_name[food_name["Food_Description_EN"].str.contains(
    "Chicken, broiler, breast, meat and skin, roasted", case=False, na=False, regex=False
)]
chicken_code = int(chicken_match.iloc[0]["Food_Code"])
chicken_desc = chicken_match.iloc[0]["Food_Description_EN"]
print(f"Food_Code: {chicken_code}  —  {chicken_desc}")

Food_Code: 839  —  Chicken, broiler, breast, meat and skin, roasted


In [17]:
# Filter Nutrient_Amount to just this food, then merge with Nutrient_Name
# to get readable names. This merge is exactly what src/calculator.py does
# for every ingredient in a recipe.
chicken_nutrients = (
    nutrient_amount[nutrient_amount["Food_Code"] == chicken_code]
    .merge(nutrient_name, on="Nutrient_Code")
)

# Show a readable panel: name, amount, unit — sorted by amount descending
panel = chicken_nutrients[["Nutrient_Name_EN", "Nutrient_Amount", "Nutrient_Unit"]]
panel = panel.sort_values("Nutrient_Amount", ascending=False)
print(f"Nutrient panel for: {chicken_desc}")
print(f"({len(panel)} nutrients measured)")
panel.head(20)

Nutrient panel for: Chicken, broiler, breast, meat and skin, roasted
(106 nutrients measured)


,Nutrient_Name_EN,Nutrient_Amount,Nutrient_Unit
16,Energy (kilojoules),824.00000,kilojoule
24,Potassium,245.00000,Milligram
23,Phosphorus,214.00000,Milligram
4,Energy (kilocalories),197.00000,kilocalorie
73,Cholesterol,84.00000,Milligram
49,"Choline, total",72.80000,Milligram
25,Sodium,71.00000,Milligram
11,Moisture,62.44000,Gram
0,Protein,29.80000,Gram
31,Retinol activity equivalents,28.00000,Microgram


### The core arithmetic: per-100g → per-recipe-grams

Every calculation in the app starts from one formula:

```
nutrient_from_ingredient = grams_used × (Nutrient_Amount / 100)
```

Because `Nutrient_Amount` is per 100 g, dividing by 100 gives the
amount per 1 g, and multiplying by the grams used scales it up. Let's
do this by hand for protein.

In [18]:
# How much protein is in 75 g of chicken breast (roasted)?
protein_row = chicken_nutrients[chicken_nutrients["Nutrient_Code"] == 203]
protein_per_100g = float(protein_row.iloc[0]["Nutrient_Amount"])
grams_used = 75

protein_in_75g = grams_used * (protein_per_100g / 100)
print(f"Chicken breast protein: {protein_per_100g:.1f} g per 100 g")
print(f"Protein in {grams_used} g: {protein_in_75g:.1f} g")
print(f"  (formula: {grams_used} × {protein_per_100g} / 100 = {protein_in_75g:.1f})")

Chicken breast protein: 29.8 g per 100 g
Protein in 75 g: 22.3 g
  (formula: 75 × 29.8 / 100 = 22.3)


### Pivot: long → wide for humans

Long format is great for storage; wide format is great for reading.
`pivot` turns "one row per food–nutrient pair" into "one row per food,
one column per nutrient" — a table you can scan across.

In [19]:
# Pick 5 foods and 6 nutrients, build a wide matrix.
sample_foods = food_name[food_name["Food_Description_EN"].str.contains(
    "Chicken, broiler, breast, meat and skin, roasted|Banana, raw|"
    "Grains, rice, white, long-grain, regular, dry|Milk, fluid, partly skimmed, 2%|"
    "Broccoli, raw",
    case=False, na=False, regex=True  # regex=True because we used | as OR
)].copy()

# Get their codes
sample_codes = sample_foods["Food_Code"].tolist()
sample_descs = dict(zip(sample_foods["Food_Code"], sample_foods["Food_Description_EN"]))

# Key nutrient codes
key_codes = [203, 204, 205, 208, 255, 291]  # protein, fat, carb, energy, water, fibre
key_code_names = dict(zip(
    key_codes,
    ["Protein_g", "Fat_g", "Carb_g", "Energy_kcal", "Water_g", "Fibre_g"]
))

# Filter + merge
sample_amounts = (
    nutrient_amount[
        (nutrient_amount["Food_Code"].isin(sample_codes)) &
        (nutrient_amount["Nutrient_Code"].isin(key_codes))
    ]
    .merge(nutrient_name[["Nutrient_Code", "Nutrient_Name_EN"]], on="Nutrient_Code")
)

# Pivot: rows = food, columns = nutrient
wide = sample_amounts.pivot_table(
    index="Food_Code",
    columns="Nutrient_Name_EN",
    values="Nutrient_Amount",
)
# Replace food codes with descriptions for readability
wide.index = [sample_descs.get(c, c) for c in wide.index]
print("Nutrient amounts per 100 g (wide format):")
wide

Nutrient amounts per 100 g (wide format):


Nutrient_Name_EN,"Carbohydrate, total (by difference)",Energy (kilocalories),Fat (total lipids),"Fibre, total dietary",Moisture,Protein
"Milk, fluid, partly skimmed, 2% M.F.",4.38,47.0,1.839,0.00,89.76,3.31
"Chicken, broiler, breast, meat and skin, roasted",0.00,197.0,7.780,0.00,62.44,29.80
"Banana, raw",22.84,89.0,0.330,1.74,74.91,1.09
"Broccoli, raw",6.64,34.0,0.370,2.40,89.30,2.82
"Grains, rice, white, long-grain, regular, dry",79.95,365.0,0.660,0.97,11.62,7.13
"Pepper, banana, raw",5.35,27.0,0.450,3.40,91.81,1.66


### Your turn 4

Build the nutrient panel (the merge from earlier) for a food you
actually eat. Pick any food, find its `Food_Code`, filter
`nutrient_amount`, merge with `nutrient_name`, and read off its sodium
per 100 g.

In [20]:
# Your code here:
# my_food = food_name[food_name["Food_Description_EN"].str.contains("...", case=False, na=False, regex=False)]
# my_code = int(my_food.iloc[0]["Food_Code"])
# my_panel = nutrient_amount[nutrient_amount["Food_Code"] == my_code].merge(nutrient_name, on="Nutrient_Code")
# sodium = my_panel[my_panel["Nutrient_Code"] == 307]
# print(sodium[["Nutrient_Name_EN", "Nutrient_Amount", "Nutrient_Unit"]])

## Part 5 — Measures: how "1 cup" becomes grams

CNF stores nutrient amounts per 100 g. But RDs and recipes think in
household measures: "1 cup of rice", "1 tablespoon of oil". To bridge
that, CNF provides a three-table chain:

1. **`Measure_Name.csv`** — descriptions like "1 cup", "1 tablespoon"
2. **`Measure_Weight_Conversion.csv`** — for each food, what a measure
   weighs in grams (e.g. "1 cup of cooked rice = 158 g")
3. **`Measure_Type.csv`** — 3 rows classifying measures:
   - Code **6** = User-defined (the household measures we want)
   - Code **3** = Refuse (bones, peels — not edible)
   - Code **9** = Yield (cooking conversion factors)

The app's `src/measures.py` automates this chain. Here you'll do it by
hand.

In [21]:
measure_name = tables["measure_name"]
measure_weight = tables["measure_weight_conversion"]
measure_type = tables["measure_type"]

print("Measure_Type (all 3 rows):")
print(measure_type.to_string(index=False))

Measure_Type (all 3 rows):
 Measure_Type_Code Measure_Type_Description_EN Measure_Type_Description_FR
                 3                      Refuse      Portion non comestible
                 6                User-defined    Défini par l'utilisateur
                 9                       Yield                   Rendement


In [22]:
# For our chicken breast, find its household measures and their gram weights.
# Filter to Measure_Type_Code == 6 (user-defined household measures only).
chicken_measures = (
    measure_weight[
        (measure_weight["Food_Code"] == chicken_code) &
        (measure_weight["Measure_Type_Code"] == 6)
    ]
    .merge(measure_name, on="Measure_Code")
)

chicken_measures[["Measure_Description_and_Unit_EN", "Measure_Weight_Conversion"]]

,Measure_Description_and_Unit_EN,Measure_Weight_Conversion
0,1 breast,196.000
1,100 ml chopped or diced,57.058
2,250 ml chopped or diced,142.645
3,100 g,100.000
4,1 food guide serving = 75g,75.000


In [23]:
# Now use a measure to compute protein. Say the chicken is "1 breast (172 g)":
# find that measure, get its gram weight, then apply the per-100g formula.
breast_measure = chicken_measures[
    chicken_measures["Measure_Description_and_Unit_EN"].str.contains(
        "breast", case=False, na=False, regex=False
    )
]
if len(breast_measure) > 0:
    grams_per_breast = float(breast_measure.iloc[0]["Measure_Weight_Conversion"])
    protein_per_100g = float(
        chicken_nutrients[chicken_nutrients["Nutrient_Code"] == 203].iloc[0]["Nutrient_Amount"]
    )
    protein_per_breast = grams_per_breast * (protein_per_100g / 100)
    print(f"1 chicken breast ≈ {grams_per_breast:.0f} g")
    print(f"Protein per 100 g: {protein_per_100g:.1f} g")
    print(f"Protein per breast: {protein_per_breast:.1f} g")
else:
    print("No 'breast' measure found for this food. Available measures:")
    print(chicken_measures["Measure_Description_and_Unit_EN"].tolist())

1 chicken breast ≈ 196 g
Protein per 100 g: 29.8 g
Protein per breast: 58.4 g


### Your turn 5

Find the gram weight of "1 cup" of milk (search `Food_Name` for
"Milk, fluid, 2% M.F.", then look up its measures). Sanity-check: a cup
of milk should weigh roughly 250 g (milk is mostly water, and water is
~1 g/mL).

In [24]:
# Your code here:
# milk = food_name[food_name["Food_Description_EN"].str.contains("Milk, fluid, 2%", case=False, na=False, regex=False)]
# milk_code = int(milk.iloc[0]["Food_Code"])
# milk_measures = measure_weight[(measure_weight["Food_Code"] == milk_code) & (measure_weight["Measure_Type_Code"] == 6)].merge(measure_name, on="Measure_Code")
# cup = milk_measures[milk_measures["Measure_Description_and_Unit_EN"].str.contains("cup", case=False, na=False, regex=False)]
# print(cup[["Measure_Description_and_Unit_EN", "Measure_Weight_Conversion"]])

## Part 6 — Where the values come from

Every row in `Nutrient_Amount.csv` carries a `Nutrient_Source_Code`
that says *where that value came from*. This matters clinically: you
want to know whether a number was analysed in a lab, copied from USDA,
or calculated. The `Nutrient_Source.csv` table decodes the numbers.

This is the section that answers the question "is CNF just USDA?" with
data. The app's CONTEXT.md states ~55.4% of values are "No change from
USDA" — let's verify that ourselves.

In [25]:
nutrient_source = tables["nutrient_source"]
print("Nutrient source codes and descriptions:")
print(nutrient_source.to_string(index=False))

Nutrient source codes and descriptions:
 Nutrient_Source_Code                                                                                   Nutrient_Source_Description_EN                                                                                        Nutrient_Source_Description_FR
                    0                                                                                              No change from USDA                                                                                      Provient intégralement de l'USDA
                    1                                                         Nutrient levels changed to meet the canadian regulations                                            Teneur en nutriments modifiée pour satisfaire la réglementation canadienne
                    2                                                                    Nutrient calculated from data other than USDA                                                                  N

In [26]:
# Join every nutrient amount to its source description, then count.
amounts_with_source = nutrient_amount.merge(
    nutrient_source, on="Nutrient_Source_Code", how="left"
)

source_counts = amounts_with_source["Nutrient_Source_Description_EN"].value_counts()
source_pct = amounts_with_source["Nutrient_Source_Description_EN"].value_counts(normalize=True) * 100

print("Where CNF nutrient values come from:")
summary = pd.DataFrame({"count": source_counts, "percent": source_pct.round(1)})
print(summary.to_string())

Where CNF nutrient values come from:
                                                                                                                   count  percent
Nutrient_Source_Description_EN                                                                                                   
No change from USDA                                                                                               313311     55.4
Nutrient analyzed in a Canadian government lab                                                                     76332     13.5
Nutrient value is an assumed zero                                                                                  65887     11.7
Calculated from analytical Canadian data                                                                           33184      5.9
Nutrient calculated from USDA data                                                                                 14897      2.6
Calculated using a recipe                            

In [27]:
# The headline number: what fraction is "No change from USDA"?
usda_pct = source_pct.get("No change from USDA", 0)
print(f"'No change from USDA': {usda_pct:.1f}% of {len(nutrient_amount):,} values")
print()
print("So CNF is a MERGED database — Health Canada took USDA values,")
print("modified some for Canadian fortification regulations, analysed")
print("others in Canadian labs, and calculated the rest. Doing that")
print("merge again from outside would do it worse.")

'No change from USDA': 55.4% of 565,409 values

So CNF is a MERGED database — Health Canada took USDA values,
modified some for Canadian fortification regulations, analysed
others in Canadian labs, and calculated the rest. Doing that
merge again from outside would do it worse.


In [28]:
# Does the source mix differ by food group? Let's check one group.
# First, get food codes for a group, then see their nutrient sources.
food_with_group = food_name.merge(tables["food_group"], on="CNF_Food_Group_Code", how="left")

# Pick "Vegetables and Vegetable Products" (a common group)
veg_group = food_with_group[
    food_with_group["CNF_Food_Group_Description_EN"].str.contains(
        "Vegetables", case=False, na=False, regex=False
    )
]
veg_codes = veg_group["Food_Code"].tolist()

veg_amounts = amounts_with_source[amounts_with_source["Food_Code"].isin(veg_codes)]
veg_source_pct = veg_amounts["Nutrient_Source_Description_EN"].value_counts(normalize=True) * 100

print(f"Source mix for 'Vegetables' foods ({len(veg_codes)} foods):")
print(veg_source_pct.round(1).to_string())

Source mix for 'Vegetables' foods (790 foods):
Nutrient_Source_Description_EN
No change from USDA                                                                                                 84.0
Nutrient value is an assumed zero                                                                                    9.4
Nutrient calculated from USDA data                                                                                   2.7
Calculated using a recipe                                                                                            1.4
Nutrient calculated from data other than USDA                                                                        0.9
Provisional data                                                                                                     0.7
Nutrient imputed from a similar USDA food                                                                            0.2
Nutrient analyzed in a Canadian government lab                             

### Your turn 6

What fraction of all nutrient values were "analyzed in a Canadian
government lab"? (The source description contains "analyzed in a
Canadian government lab" — use the `amounts_with_source` DataFrame
from above.)

In [29]:
# Your code here:
# lab_pct = (amounts_with_source["Nutrient_Source_Description_EN"] == "...").mean() * 100
# print(f"{lab_pct:.1f}%")

## Part 7 — Data quality, the RD questions

Not every food has every nutrient measured. When the app builds a
recipe, it shows a **Coverage** column — "how many of this recipe's
ingredients actually had CNF data for this nutrient?" — because a
missing value is visible (you know you don't know), while a silently
fabricated zero is not.

Here we'll compute, for the app's key nutrients, how many of the ~5,993
foods have a value. This is the kind of question an RD asks before
trusting a database: "how complete is it for the things I care about?"

In [30]:
# The app's key nutrient codes (from the registry we loaded in Part 3):
app_codes = registry["code"].tolist()
app_names = dict(zip(registry["code"], registry["label"]))

# For each tracked nutrient, how many foods have a value?
total_foods = food_name["Food_Code"].nunique()
coverage_rows = []
for code in app_codes:
    n_with_value = nutrient_amount[nutrient_amount["Nutrient_Code"] == code]["Food_Code"].nunique()
    pct = 100 * n_with_value / total_foods
    coverage_rows.append({
        "nutrient": app_names.get(code, code),
        "code": code,
        "foods_with_value": n_with_value,
        "coverage_pct": round(pct, 1),
    })

coverage_df = pd.DataFrame(coverage_rows).sort_values("coverage_pct", ascending=False)
print(f"Coverage across {total_foods} foods:")
print(coverage_df.to_string(index=False))

Coverage across 5993 foods:
        nutrient  code  foods_with_value  coverage_pct
          Energy   208              5993         100.0
         Protein   203              5993         100.0
Water (moisture)   255              5993         100.0
    Carbohydrate   205              5993         100.0
             Fat   204              5993         100.0
            Iron   303              5954          99.3
         Calcium   301              5954          99.3
          Sodium   307              5950          99.3
      Phosphorus   305              5853          97.7
       Potassium   306              5835          97.4
       Magnesium   304              5794          96.7
            Zinc   309              5788          96.6
           Fibre   291              5774          96.3
   Saturated Fat   606              5767          96.2
     Cholesterol   601              5744          95.8
     Vitamin B12   418              5594          93.3
       Vitamin D   328              5

In [31]:
# Which foods have the most lab analyses behind them?
# The 'Observations' column in Nutrient_Amount tells us how many
# analyses contributed to each value.
most_observed = (
    nutrient_amount.groupby("Food_Code")["Observations"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
most_observed = most_observed.merge(food_name[["Food_Code", "Food_Description_EN"]], on="Food_Code")
print("Top 10 foods by total observation count:")
print(most_observed[["Food_Description_EN", "Observations"]].to_string(index=False))

Top 10 foods by total observation count:
                                                        Food_Description_EN  Observations
Beans, snap (Italian, green or yellow), canned, solids and liquid, unsalted       15951.0
                                                 Sweets, syrup, maple, bulk       11379.0
                           Peas, green, canned, solids and liquid, unsalted        6871.0
                                   Milk, fluid, human (breast milk), mature        6736.0
        Peach, canned halves or slices, heavy syrup pack, solids and liquid        5667.0
                                                     Cherry, sour, red, raw        5128.0
                         Alcohol, table wine, all (11.5% alcohol by volume)        5097.0
                           Pineapple, canned, juice pack, solids and liquid        5092.0
                                Nuts, almonds, dried, unblanched, unroasted        4916.0
                                   Lamb, cubed for stew or 

### Your turn 7

Pick one nutrient you care about clinically (e.g. iron, code 303, or
fibre, code 291). Find its coverage percentage — what fraction of the
~5,993 foods have a value for it?

In [32]:
# Your code here:
# my_code = 303  # iron
# n = nutrient_amount[nutrient_amount["Nutrient_Code"] == my_code]["Food_Code"].nunique()
# print(f"Coverage: {n}/{total_foods} = {100*n/total_foods:.1f}%")

## Part 8 — Bridge to the app

Everything above, the app has productionized. `src/data_loader.py`
loads the CSVs (with the BOM fix built in). `src/food_search.py`
implements the three-layer search that fixes the "wild rice" problem.
`src/calculator.py` does the per-100g scaling and merge for every
ingredient in a recipe.

Here we'll use the app's own modules to do the same tasks in one line
each, and then do a **capstone**: hand-compute the kcal for a
2-ingredient mini-blend, then run the same ingredients through the
app's calculator and confirm the numbers agree.

In [33]:
import sys
sys.path.insert(0, "..")  # so we can import the src/ package

from src.data_loader import load_all
from src.food_search import build_index, search_foods
from src.calculator import calculate_profile
from src.models import Ingredient, Recipe

# One-liner: load all CNF tables (the app's loader, with BOM handling built in)
app_tables = load_all()
for name, df in app_tables.items():
    print(f"{name:30s}  {df.shape[0]:>8,} rows × {df.shape[1]} cols")

food_name                          5,993 rows × 12 cols
nutrient_name                        173 rows × 7 cols
nutrient_amount                  565,409 rows × 7 cols
measure_name                       1,494 rows × 3 cols
measure_type                           3 rows × 3 cols
measure_weight_conversion         29,868 rows × 5 cols
food_group                            23 rows × 3 cols


In [34]:
# The three-layer search: "wild rice" now succeeds.
# build_index() pre-tokenises the food names; search_foods() runs the
# three layers (all-words, typo-tolerant, synonyms) and returns ranked
# candidates plus a note explaining how it interpreted your query.
index = build_index(app_tables["food_name"])
result = search_foods("wild rice", index)
print(f"Match type: {result.match_type}")
print(f"Note: {result.note}")
print(f"Results: {len(result.matches)}")
result.matches[["Food_Code", "Food_Description_EN"]].head(5)

Match type: direct
Note: 
Results: 3


,Food_Code,Food_Description_EN
3299,4449,"Grains, rice, wild, dry"
3300,4450,"Grains, rice, wild, cooked"
5699,7701,"Rice, white and wild, flavoured, unprepared"


In [35]:
# Capstone: hand-compute kcal for a 2-ingredient mini-blend, then
# check it against the app's calculator.

# Ingredients: 150 g chicken breast + 200 g "Water, municipal"
na = app_tables["nutrient_amount"]
fn = app_tables["food_name"]

# Find the food codes
chicken_row = fn[fn["Food_Description_EN"].str.contains(
    "Chicken, broiler, breast, meat and skin, roasted", case=False, na=False, regex=False
)].iloc[0]
water_row = fn[fn["Food_Description_EN"].str.contains(
    "Water, municipal", case=False, na=False, regex=False
)].iloc[0]

chicken_code = int(chicken_row["Food_Code"])
water_code = int(water_row["Food_Code"])
print(f"Chicken: code {chicken_code} — {chicken_row['Food_Description_EN']}")
print(f"Water:   code {water_code} — {water_row['Food_Description_EN']}")

Chicken: code 839 — Chicken, broiler, breast, meat and skin, roasted
Water:   code 2933 — Water, municipal


In [36]:
# Hand computation: kcal = grams × (energy_per_100g / 100), summed.
chicken_energy = float(
    na[(na["Food_Code"] == chicken_code) & (na["Nutrient_Code"] == 208)].iloc[0]["Nutrient_Amount"]
)
water_energy = float(
    na[(na["Food_Code"] == water_code) & (na["Nutrient_Code"] == 208)].iloc[0]["Nutrient_Amount"]
)

chicken_kcal = 150 * (chicken_energy / 100)
water_kcal = 200 * (water_energy / 100)
hand_total_kcal = chicken_kcal + water_kcal

print(f"Chicken: {chicken_energy:.1f} kcal/100g → 150 g = {chicken_kcal:.1f} kcal")
print(f"Water:   {water_energy:.1f} kcal/100g → 200 g = {water_kcal:.1f} kcal")
print(f"Hand total: {hand_total_kcal:.1f} kcal")

Chicken: 197.0 kcal/100g → 150 g = 295.5 kcal
Water:   0.0 kcal/100g → 200 g = 0.0 kcal
Hand total: 295.5 kcal


In [37]:
# App computation: same ingredients through the calculator.
recipe = Recipe(
    name="Mini blend",
    ingredients=[
        Ingredient(food_code=chicken_code, food_description="Chicken breast", grams=150),
        Ingredient(food_code=water_code, food_description="Water, municipal", grams=200),
    ],
    measured_final_volume_mL=350,  # measured, not computed
)

profile = calculate_profile(recipe, na)
app_total_kcal = profile.nutrient_totals.get("energy_kcal", 0.0)

print(f"App total: {app_total_kcal:.1f} kcal")
print(f"Hand total: {hand_total_kcal:.1f} kcal")
print(f"Match: {abs(app_total_kcal - hand_total_kcal) < 0.01}")
print()
print(f"kcal/mL:  {profile.kcal_per_mL:.3f}")
print(f"protein/mL: {profile.protein_per_mL:.3f}")

App total: 295.5 kcal


Hand total: 295.5 kcal
Match: True

kcal/mL:  0.844
protein/mL: 0.128


### Your turn 8 (free play)

Swap in your own two ingredients. Search for them with `search_foods`,
find their codes, build a `Recipe`, and run `calculate_profile`. Try a
food you'd actually put in a blend.

In [38]:
# Your code here — free play, no single right answer:
# result = search_foods("...", index)
# print(result.matches[["Food_Code", "Food_Description_EN"]].head())
# # pick a code, build a Recipe, run calculate_profile...

## Answers

Worked answers for each **Your turn** cell. Try the exercise yourself
first — the learning happens in the struggle, not in reading this.

**Your turn 0** — Load Food_Name.csv:

In [39]:
food_name = pd.read_csv(DATA_DIR / "Food_Name.csv", encoding="utf-8-sig")
print(f"Shape: {food_name.shape}")
print(f"First column: {food_name.columns[0]}")

Shape: (5993, 12)
First column: Food_Code


**Your turn 1** — Two tables with Nutrient_Source_Code:

In [40]:
for name, df in tables.items():
    if "Nutrient_Source_Code" in df.columns:
        print(f"  {name}")
# Answer: Nutrient_Amount and Nutrient_Source

  nutrient_amount
  nutrient_source


**Your turn 2** — Cheese foods and their group:

In [41]:
cheese = food_name[food_name["Food_Description_EN"].str.contains(
    "cheese", case=False, na=False, regex=False
)]
print(f"Foods containing 'cheese': {len(cheese)}")

cheese_with_group = cheese.merge(tables["food_group"], on="CNF_Food_Group_Code", how="left")
print(cheese_with_group["CNF_Food_Group_Description_EN"].value_counts().head(3))

Foods containing 'cheese': 328
CNF_Food_Group_Description_EN
Dairy and Egg Products    126
Mixed Dishes               80
Fast Foods                 57
Name: count, dtype: int64


**Your turn 3** — Sugars code and unit:

In [42]:
sugars = nutrient_name[nutrient_name["Nutrient_Name_EN"].str.contains(
    "Sugars", case=False, na=False, regex=False
)]
print(sugars[["Nutrient_Code", "Nutrient_Name_EN", "Nutrient_Unit"]].to_string(index=False))

 Nutrient_Code Nutrient_Name_EN Nutrient_Unit
           269    Sugars, total          Gram


**Your turn 4** — Nutrient panel for a food you eat (example: banana):

In [43]:
my_food = food_name[food_name["Food_Description_EN"].str.contains(
    "Banana, raw", case=False, na=False, regex=False
)]
my_code = int(my_food.iloc[0]["Food_Code"])
my_panel = nutrient_amount[nutrient_amount["Food_Code"] == my_code].merge(
    nutrient_name, on="Nutrient_Code"
)
sodium = my_panel[my_panel["Nutrient_Code"] == 307]
print(f"Food: {my_food.iloc[0]['Food_Description_EN']}")
print(sodium[["Nutrient_Name_EN", "Nutrient_Amount", "Nutrient_Unit"]].to_string(index=False))

Food: Banana, raw
Nutrient_Name_EN  Nutrient_Amount Nutrient_Unit
          Sodium              1.0     Milligram


**Your turn 5** — Gram weight of 1 cup of 2% milk:

In [44]:
milk = food_name[food_name["Food_Description_EN"].str.contains(
    "Milk, fluid, partly skimmed, 2%", case=False, na=False, regex=False
)]
milk_code = int(milk.iloc[0]["Food_Code"])
milk_measures = (
    measure_weight[
        (measure_weight["Food_Code"] == milk_code) &
        (measure_weight["Measure_Type_Code"] == 6)
    ]
    .merge(measure_name, on="Measure_Code")
)
cup = milk_measures[milk_measures["Measure_Description_and_Unit_EN"].str.contains(
    "cup", case=False, na=False, regex=False
)]
print(cup[["Measure_Description_and_Unit_EN", "Measure_Weight_Conversion"]].to_string(index=False))
# Should be ~250 g — milk is mostly water (~1 g/mL).

Empty DataFrame
Columns: [Measure_Description_and_Unit_EN, Measure_Weight_Conversion]
Index: []


**Your turn 6** — Fraction analyzed in a Canadian government lab:

In [45]:
lab_pct = (
    amounts_with_source["Nutrient_Source_Description_EN"]
    .str.contains("analyzed in a Canadian government lab", case=False, na=False)
    .mean()
    * 100
)
print(f"Analyzed in a Canadian government lab: {lab_pct:.1f}%")

Analyzed in a Canadian government lab: 13.5%


**Your turn 7** — Coverage for one nutrient (example: iron, code 303):

In [46]:
my_code = 303  # iron
n = nutrient_amount[nutrient_amount["Nutrient_Code"] == my_code]["Food_Code"].nunique()
print(f"Iron coverage: {n}/{total_foods} foods = {100*n/total_foods:.1f}%")

Iron coverage: 5954/5993 foods = 99.3%


## Cheat sheet

The idioms you'll reach for again and again:

```python
# Load a CNF CSV (utf-8-sig handles the BOM, safe for all files):
df = pd.read_csv("cnf_fcen_all-files-data_2026/Nutrient_Name.csv",
                 encoding="utf-8-sig")

# Case-insensitive substring search (regex=False is the safe default):
matches = food_name[food_name["Food_Description_EN"].str.contains(
    "chicken", case=False, na=False, regex=False
)]

# Merge two tables on a shared key (like VLOOKUP):
panel = nutrient_amount.merge(nutrient_name, on="Nutrient_Code")

# The core per-100g scaling formula:
nutrient_from_ingredient = grams * (Nutrient_Amount / 100)

# Pivot long → wide for a readable matrix:
wide = df.pivot_table(index="Food_Code", columns="Nutrient_Name_EN",
                      values="Nutrient_Amount")

# Provenance breakdown (where values come from):
pct = df["Nutrient_Source_Description_EN"].value_counts(normalize=True) * 100

# The app's one-liner equivalents:
from src.data_loader import load_all
tables = load_all()

from src.food_search import build_index, search_foods
index = build_index(tables["food_name"])
result = search_foods("wild rice", index)
```

**Key nutrient codes to remember:**
protein 203, fat 204, carbohydrate 205, energy 208 (kcal),
water/moisture 255, fibre 291, sodium 307, potassium 306,
calcium 301, iron 303.

**The one rule:** all CNF nutrient amounts are per 100 g of edible
food. Everything else follows from that.